# Public MPN Expression Analysis — GSE168368

This notebook extends the JAK2 V617F project with a real public human MPN expression dataset. NCBI GEO GSE168368 contains 29 MEP samples spanning PV, ET and healthy controls. The dataset is transcriptomic evidence and is **not** a direct JAK2 V617F variant-calling dataset.

Source: NCBI GEO GSE168368 / BioProject PRJNA707039.

In [ ]:
from pathlib import Path
import sys
import urllib.request
import gzip
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
DATA = ROOT / 'data'
RAW = DATA / 'raw'
PROCESSED = DATA / 'processed'
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
URL = 'https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE168368&file=GSE168368_gene_count_matrix.csv.gz&format=file'
MATRIX = RAW / 'GSE168368_gene_count_matrix.csv.gz'


## 1. Retrieve the public processed matrix

The matrix is downloaded directly from NCBI GEO. It is intentionally not committed to the repository; provenance is recorded instead.

In [ ]:
if not MATRIX.exists():
    urllib.request.urlretrieve(URL, MATRIX)
print(MATRIX, MATRIX.stat().st_size, 'bytes')


In [ ]:
counts = pd.read_csv(MATRIX, compression='gzip')
counts.head()


## 2. Verify sample structure

The GEO series contains 9 PV, 12 ET and 8 healthy-control (NC) samples.

In [ ]:
sample_cols = [c for c in counts.columns if str(c).startswith(('PV','ET','NC'))]
group_counts = pd.Series([str(c)[:2] for c in sample_cols]).value_counts().reindex(['PV','ET','NC'])
assert len(sample_cols) == 29
group_counts


In [ ]:
ax = group_counts.plot(kind='bar', figsize=(7,4), title='GSE168368 sample groups')
ax.set_xlabel('Group')
ax.set_ylabel('Number of samples')
plt.tight_layout()
plt.show()


## 3. Identify JAK2 expression

The code below searches common gene-identifier columns and gene symbols. Because this is expression data, the output is interpreted as **JAK2 expression**, not as direct evidence of the V617F genotype.

In [ ]:
candidate_cols = [c for c in counts.columns if str(c).lower() in {'gene','gene_symbol','symbol','geneid','gene_id','genes'}]
candidate_cols


In [ ]:
gene_col = candidate_cols[0] if candidate_cols else counts.columns[0]
jak2 = counts[counts[gene_col].astype(str).str.upper().eq('JAK2')].copy()
if jak2.empty:
    print('JAK2 was not found by exact symbol in the selected identifier column; inspect the matrix annotation before proceeding.')
else:
    values = jak2[sample_cols].iloc[0].astype(float)
    expr = pd.DataFrame({'sample': values.index, 'expression': values.values})
    expr['group'] = expr['sample'].str[:2]
    display(expr.groupby('group')['expression'].agg(['count','mean','median','std']))


In [ ]:
if 'expr' in globals():
    ax = expr.boxplot(column='expression', by='group', figsize=(7,5))
    plt.suptitle('')
    ax.set_title('JAK2 expression across GSE168368 groups')
    ax.set_xlabel('Group')
    ax.set_ylabel('Count value')
    plt.tight_layout()
    plt.show()


## 4. Interpretation

This analysis adds real public MPN biological context to the sequence-level JAK2 V617F workflow. It does not infer individual patient genotype from expression. Direct JAK2 V617F detection would require appropriate variant-level sequencing or genotyping data.

The canonical variant remains NM_004972.4:c.1849G>T → p.Val617Phe (rs77375493).

## 5. Reproducibility record

Record the NCBI accession, retrieval date, matrix checksum, software environment and repository commit whenever the notebook is executed for publication.

**Source:** GSE168368 / PRJNA707039 / SRP310755.